In [ ]:
IMPORT LIBRARY

In [2]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_text, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)

OUT = "hasil_rf"
os.makedirs(OUT, exist_ok=True)
np.random.seed(42)

In [ ]:
PEMBUATAN DATASET

In [3]:
n_samples = 250
dist_0 = np.random.normal(100, 40, n_samples)
idle_0 = np.random.normal(60, 15, n_samples)
iso_0 = np.random.normal(100, 30, n_samples)
dist_1 = np.random.normal(80, 25, n_samples)
idle_1 = np.random.normal(80, 10, n_samples)
iso_1 = np.random.normal(150, 40, n_samples)

df = pd.DataFrame({
    'total_distance': np.concatenate([dist_0, dist_1]),
    'idle_duration': np.concatenate([idle_0, idle_1]),
    'isolation_index': np.concatenate([iso_0, iso_1]),
    'status': np.concatenate([np.zeros(n_samples), np.ones(n_samples)])
})
df.to_csv('dataset_ayam_simple.csv', index=False)

X = df.drop(columns=['status'])
y = df['status']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
FEATURES = list(X.columns)
CLASS_NAMES = ['Sehat (0)', 'Pasif (1)']

In [ ]:
PERBANDINGAN n_estimator dan max_depth

In [4]:
N_ESTIMATORS = [1, 5, 10, 25, 50, 100, 200]
MAX_DEPTHS = [1, 2, 3, 5, 7, 10, None]       # None = tanpa batas
depth_label = lambda d: "None" if d is None else str(d)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for n in N_ESTIMATORS:
    for d in MAX_DEPTHS:
        m = RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42)
        m.fit(X_train, y_train)
        p = m.predict(X_test)
        cv_f1 = cross_val_score(
            RandomForestClassifier(n_estimators=n, max_depth=d, random_state=42),
            X_train, y_train, cv=cv, scoring='f1')
        rows.append({
            'n_estimators': n,
            'max_depth': depth_label(d),
            'accuracy': accuracy_score(y_test, p),
            'precision': precision_score(y_test, p, zero_division=0),
            'recall': recall_score(y_test, p, zero_division=0),
            'f1_score': f1_score(y_test, p, zero_division=0),
            'cv_f1_mean': cv_f1.mean(),
            'cv_f1_std': cv_f1.std(),
        })
res = pd.DataFrame(rows)
res.to_csv(f"{OUT}/tabel_perbandingan.csv", index=False)

pd.set_option('display.width', 200)
print("=" * 70)
print("1 & 3. TABEL PERBANDINGAN (data test + 5-fold CV pada data train)")
print("=" * 70)
print(res.round(4).to_string(index=False))

1 & 3. TABEL PERBANDINGAN (data test + 5-fold CV pada data train)
 n_estimators max_depth  accuracy  precision  recall  f1_score  cv_f1_mean  cv_f1_std
            1         1      0.65     0.6119    0.82    0.7009      0.7042     0.0252
            1         2      0.76     0.6912    0.94    0.7966      0.8323     0.0269
            1         3      0.77     0.7368    0.84    0.7850      0.8110     0.0945
            1         5      0.80     0.7586    0.88    0.8148      0.8301     0.0486
            1         7      0.81     0.8163    0.80    0.8081      0.8226     0.0619
            1        10      0.82     0.7667    0.92    0.8364      0.8069     0.0912
            1      None      0.82     0.7667    0.92    0.8364      0.8204     0.0634
            5         1      0.75     0.7193    0.82    0.7664      0.8061     0.0613
            5         2      0.85     0.8182    0.90    0.8571      0.8456     0.0296
            5         3      0.85     0.8431    0.86    0.8515      0.8332

In [ ]:
Grafik Perbandingan

In [5]:
depth_order = [depth_label(d) for d in MAX_DEPTHS]

# 3a. Heatmap 4 metrik
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, metric in zip(axes.ravel(), ['accuracy', 'precision', 'recall', 'f1_score']):
    pv = res.pivot(index='n_estimators', columns='max_depth', values=metric)[depth_order]
    im = ax.imshow(pv.values, cmap='YlGnBu', aspect='auto')
    ax.set_xticks(range(len(depth_order))); ax.set_xticklabels(depth_order)
    ax.set_yticks(range(len(pv.index))); ax.set_yticklabels(pv.index)
    ax.set_xlabel('max_depth'); ax.set_ylabel('n_estimators')
    ax.set_title(metric.capitalize())
    for i in range(pv.shape[0]):
        for j in range(pv.shape[1]):
            ax.text(j, i, f"{pv.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(f"{OUT}/grafik_heatmap_metrik.png", dpi=150)
plt.close()

# 3b. Garis: metrik vs n_estimators (per depth), dan vs depth (per n_estimators)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for d in depth_order:
    s = res[res.max_depth == d]
    axes[0].plot(s.n_estimators, s.accuracy, marker='o', label=f"depth={d}")
axes[0].set_xscale('log'); axes[0].set_xlabel('n_estimators'); axes[0].set_ylabel('Akurasi (test)')
axes[0].set_title('Akurasi vs Jumlah Pohon'); axes[0].grid(alpha=.3); axes[0].legend(fontsize=8)
for n in N_ESTIMATORS:
    s = res[res.n_estimators == n].set_index('max_depth').loc[depth_order]
    axes[1].plot(depth_order, s.accuracy, marker='o', label=f"n={n}")
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Akurasi (test)')
axes[1].set_title('Akurasi vs Kedalaman Pohon'); axes[1].grid(alpha=.3); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/grafik_akurasi_garis.png", dpi=150)
plt.close()

# 3c. Heatmap F1 hasil cross-validation (lebih stabil)
pv = res.pivot(index='n_estimators', columns='max_depth', values='cv_f1_mean')[depth_order]
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(pv.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(depth_order))); ax.set_xticklabels(depth_order)
ax.set_yticks(range(len(pv.index))); ax.set_yticklabels(pv.index)
for i in range(pv.shape[0]):
    for j in range(pv.shape[1]):
        ax.text(j, i, f"{pv.values[i, j]:.3f}", ha='center', va='center', fontsize=8)
ax.set_xlabel('max_depth'); ax.set_ylabel('n_estimators'); ax.set_title('F1-score rata-rata 5-fold CV')
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(f"{OUT}/grafik_heatmap_cv_f1.png", dpi=150)
plt.close()

In [ ]:
PEMILIHAN MODEL PALING OPTIMAL

In [6]:
# Kriteria: F1 CV tertinggi. Jika selisih <= 0.005 dianggap setara,
# pilih model paling sederhana (pohon paling sedikit, lalu paling dangkal).
best_cv = res.cv_f1_mean.max()
cand = res[res.cv_f1_mean >= best_cv - 0.005].copy()
cand['depth_num'] = cand.max_depth.apply(lambda v: 999 if v == "None" else int(v))
opt = cand.sort_values(['n_estimators', 'depth_num']).iloc[0]
opt_n = int(opt.n_estimators)
opt_d = None if opt.max_depth == "None" else int(opt.max_depth)

print("\n" + "=" * 70)
print("4. KESIMPULAN PARAMETER OPTIMAL")
print("=" * 70)
print("Top 5 berdasarkan F1 cross-validation:")
print(res.sort_values('cv_f1_mean', ascending=False).head(5).round(4).to_string(index=False))
print(f"\n>> Optimal: n_estimators={opt_n}, max_depth={opt_d}")
print(f"   CV F1={opt.cv_f1_mean:.4f} (+/- {opt.cv_f1_std:.4f}) | "
      f"Test: acc={opt.accuracy:.4f}, prec={opt.precision:.4f}, "
      f"rec={opt.recall:.4f}, f1={opt.f1_score:.4f}")


4. KESIMPULAN PARAMETER OPTIMAL
Top 5 berdasarkan F1 cross-validation:
 n_estimators max_depth  accuracy  precision  recall  f1_score  cv_f1_mean  cv_f1_std
           25         5      0.89     0.8980    0.88    0.8889      0.8742     0.0395
           50         7      0.85     0.8571    0.84    0.8485      0.8695     0.0203
          200         5      0.86     0.8750    0.84    0.8571      0.8686     0.0376
           50         5      0.86     0.8750    0.84    0.8571      0.8668     0.0310
          200         1      0.86     0.8600    0.86    0.8600      0.8656     0.0212

>> Optimal: n_estimators=25, max_depth=5
   CV F1=0.8742 (+/- 0.0395) | Test: acc=0.8900, prec=0.8980, rec=0.8800, f1=0.8889


In [ ]:
CONFUSION MATRIX

In [8]:
best_model = RandomForestClassifier(n_estimators=opt_n, max_depth=opt_d, random_state=42)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("\n" + "=" * 70)
print("5. CONFUSION MATRIX MODEL OPTIMAL")
print("=" * 70)
print(cm)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f"Confusion Matrix (n_estimators={opt_n}, max_depth={opt_d})")
plt.tight_layout()
plt.savefig(f"{OUT}/confusion_matrix_optimal.png", dpi=150)
plt.close()
joblib.dump(best_model, f"{OUT}/rf_perilaku_ayam_optimal.joblib")


5. CONFUSION MATRIX MODEL OPTIMAL
[[45  5]
 [ 6 44]]
              precision    recall  f1-score   support

   Sehat (0)       0.88      0.90      0.89        50
   Pasif (1)       0.90      0.88      0.89        50

    accuracy                           0.89       100
   macro avg       0.89      0.89      0.89       100
weighted avg       0.89      0.89      0.89       100



['hasil_rf/rf_perilaku_ayam_optimal.joblib']

In [ ]:
OPERASI PADA SETIAP POHON DAN KEDALAMAN

In [11]:
def analisis_pohon(model, nama, sampel, max_print_trees=3):
    print("\n" + "=" * 70)
    print(f"2. OPERASI PADA SETIAP POHON - {nama}")
    print("=" * 70)

    # (a) Ringkasan struktur tiap pohon
    ringkas = []
    for i, est in enumerate(model.estimators_):
        t = est.tree_
        ringkas.append({'pohon': i + 1, 'kedalaman_aktual': t.max_depth,
                        'jumlah_node': t.node_count, 'jumlah_daun': t.n_leaves,
                        'fitur_akar': FEATURES[t.feature[0]],
                        'threshold_akar': round(t.threshold[0], 2)})
    ringkas = pd.DataFrame(ringkas)
    print("\n(a) Ringkasan struktur tiap pohon:")
    print(ringkas.head(10).to_string(index=False))
    if len(ringkas) > 10:
        print(f"   ... ({len(ringkas)} pohon; ringkasan penuh di CSV)")
    ringkas.to_csv(f"{OUT}/struktur_pohon_{nama}.csv", index=False)

    # (b) Jumlah node & fitur yang dipakai split pada setiap level kedalaman
    print("\n(b) Operasi split per level kedalaman (pohon pertama):")
    t = model.estimators_[0].tree_
    depth = np.zeros(t.node_count, dtype=int)
    for node in range(t.node_count):
        for c in (t.children_left[node], t.children_right[node]):
            if c != -1:
                depth[c] = depth[node] + 1
    for lv in range(depth.max() + 1):
        nodes = np.where(depth == lv)[0]
        split = [n for n in nodes if t.children_left[n] != -1]
        leaf = len(nodes) - len(split)
        fcount = pd.Series([FEATURES[t.feature[n]] for n in split]).value_counts().to_dict()
        print(f"   Level {lv}: {len(nodes)} node | {len(split)} split, {leaf} daun | fitur: {fcount}")

    # (c) Aturan teks beberapa pohon pertama
    for i in range(min(max_print_trees, len(model.estimators_))):
        print(f"\n(c) Aturan keputusan Pohon {i + 1}:")
        print(export_text(model.estimators_[i], feature_names=FEATURES, max_depth=4))

    # (d) Jejak satu sampel melewati tiap pohon + voting
    x = sampel.values.reshape(1, -1)
    x_df = pd.DataFrame(x, columns=FEATURES)  # DataFrame: untuk model utama
    print("(d) Jejak keputusan 1 sampel uji:", dict(zip(FEATURES, sampel.round(2))))
    votes = []
    for i, est in enumerate(model.estimators_[:max_print_trees]):
        path = est.decision_path(x).indices
        t = est.tree_
        langkah = []
        for n in path:
            if t.children_left[n] == -1:
                langkah.append("DAUN")
            else:
                f = FEATURES[t.feature[n]]
                arah = "<=" if x[0, t.feature[n]] <= t.threshold[n] else ">"
                langkah.append(f"{f}{arah}{t.threshold[n]:.1f}")
        print(f"   Pohon {i + 1}: " + " -> ".join(langkah))
    for est in model.estimators_:
        votes.append(int(est.predict(x)[0]))
    v = pd.Series(votes).value_counts().to_dict()
    print(f"   Voting {len(votes)} pohon: {v} -> prediksi akhir = "
          f"{CLASS_NAMES[int(model.predict(x_df)[0])]} "
          f"(proba={model.predict_proba(x_df)[0].round(3)})")

    # (e) Feature importance
    imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
    print("\n(e) Feature importance:\n" + imp.round(4).to_string())

contoh = X_test.iloc[0]
analisis_pohon(best_model, f"optimal_n{opt_n}_d{opt_d}", contoh)

# Demo kecil: 3 pohon x kedalaman 1, 2, 3 -> memperlihatkan efek kedalaman
for d in (1, 2, 3):
    demo = RandomForestClassifier(n_estimators=3, max_depth=d, random_state=42).fit(X_train, y_train)
    analisis_pohon(demo, f"demo_n3_d{d}", contoh)
    fig, axes = plt.subplots(1, 3, figsize=(7 * 3 if d == 3 else 5 * 3, 5 if d < 3 else 6))
    for i, ax in enumerate(axes):
        plot_tree(demo.estimators_[i], feature_names=FEATURES, class_names=CLASS_NAMES,
                  filled=True, ax=ax, fontsize=7, impurity=False)
        ax.set_title(f"Pohon {i + 1} (max_depth={d})")
    plt.tight_layout()
    plt.savefig(f"{OUT}/visual_pohon_demo_d{d}.png", dpi=130)
    plt.close()


2. OPERASI PADA SETIAP POHON - optimal_n25_d5

(a) Ringkasan struktur tiap pohon:
 pohon  kedalaman_aktual  jumlah_node  jumlah_daun      fitur_akar  threshold_akar
     1                 5           45           23  total_distance          102.28
     2                 5           45           23 isolation_index          118.52
     3                 5           43           22   idle_duration           65.36
     4                 5           35           18 isolation_index          119.68
     5                 5           47           24 isolation_index          131.87
     6                 5           37           19   idle_duration           63.26
     7                 5           33           17   idle_duration           65.36
     8                 5           41           21  total_distance          102.32
     9                 5           41           21 isolation_index          131.87
    10                 5           39           20  total_distance          106.85
   .

In [ ]:
HASIL

In [10]:
print(f"\nSemua output tersimpan di folder '{OUT}/'")


Semua output tersimpan di folder 'hasil_rf/'
